# Lab 5: GPU Shade Mapping with PyCUDA — Hourly Shadow Maps of a Philadelphia Block

**Name:** Qiwen Bian  **Date:** 2026-10-06

This notebook simulates how the shadow pattern of a Philadelphia block evolves between 8:00 and 17:00,
based on the course Lab 05 notebook (*GPU programming for the urban analysis*, Xiaojiang Li) and run on a
Colab **T4 GPU** runtime. Every pixel of a 1 m digital surface model (DSM) launches a ray toward the sun;
if any surface along the ray is higher than the ray itself, the pixel is in shadow. Ten hourly shadow maps
are combined into `shadow_analysis.gif`.

**Workflow:** (1) prepare the PyCUDA modules, (2) download the course DSM tile `row11-col17.tif`
(EPSG:2272, 1 m) with `gdown` instead of mounting Google Drive, (3) run the slope kernel as a PyCUDA
warm-up, (4) run the shadow kernel once as a test, (5) homework — loop over the hourly solar positions
(NOAA solar calculator) 8 am–5 pm, save one shadow map per hour, and combine the PNGs into a GIF
| outputs: `shadow_maps/shadow_08.png … shadow_17.png`, `shadow_analysis.gif`.


## Step 1: Prepare the modules

In [ ]:
!pip install pycuda
!pip install rasterio

In [ ]:
from pycuda.compiler import SourceModule
import pycuda
from pycuda import gpuarray
from pycuda import compiler
import pycuda.autoinit             # PyCuda autoinit
import pycuda.driver as cuda       # PyCuda In, Out helpers
import os, os.path
import rasterio as rio
from osgeo import gdal
from osgeo.gdalconst import *
import numpy as np
import rasterio
import time
import math
# Load the Drive helper and mount
from google.colab import drive
import matplotlib.cm as colormap   # Library to plot
import numpy                       # Fast math library
import matplotlib.image as mpimg       # reading images to numpy arrays
import matplotlib.pyplot as plt        # to plot any graph
import scipy.ndimage as ndi            # to determine shape centrality
# matplotlib setup

%matplotlib inline
from pylab import rcParams
rcParams['figure.figsize'] = (8, 8)      # setting default size of plots

print("pycuda installed and ready to use.")

print("%d device(s) found." % cuda.Device.count())
for ordinal in range(cuda.Device.count()):
    dev = cuda.Device(ordinal)
    print ("Device #%d: %s" % (ordinal, dev.name()))
print (cuda)

## Step 2: Download and read the DSM

The course provides the DSM tile on Google Drive ([link](https://drive.google.com/file/d/1eEdw7mZ9XOGN6DNQMdiYJoDXkERV0e0n/view));
`gdown` fetches the same file directly, so no Drive mount is needed. Everything after the download is the
course notebook's original reading code.


In [ ]:

# download the course DSM from the Lab 05 Drive link (no Drive mount needed)
# https://drive.google.com/file/d/1eEdw7mZ9XOGN6DNQMdiYJoDXkERV0e0n/view
!gdown 1eEdw7mZ9XOGN6DNQMdiYJoDXkERV0e0n -O /content/row11-col17.tif

dsm_file = "/content/row11-col17.tif"
os.path.exists(dsm_file)

dsm_dataset = rio.open(dsm_file)

dsm_bounds = dsm_dataset.bounds
dsm_img = dsm_dataset.read(1)
print('The dsm bound is:', dsm_bounds)


transform = dsm_dataset.transform

# The pixel width and height (scale) can be found as follows:
pixel_width = transform[0]  # Scale in the x-direction
pixel_height = -transform[4]  # Scale in the y-direction (usually negative)

# You can calculate a unified scale if needed (average or reciprocal)
scale = 1 / pixel_width  # This will give you a scale factor


## Step 3: PyCUDA warm-up — the slope kernel

In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# Define the CUDA kernel for slope calculation
kernel_code = """
__global__ void calculate_slope(float *dsm, float *slope, int width, int height, float cell_size) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x > 0 && x < width - 1 && y > 0 && y < height - 1) {
        // Calculate gradient in x and y directions
        float dzdx = (dsm[y * width + (x + 1)] - dsm[y * width + (x - 1)]) / (2 * cell_size);
        float dzdy = (dsm[(y + 1) * width + x] - dsm[(y - 1) * width + x]) / (2 * cell_size);

        // Calculate slope
        slope[y * width + x] = atan(sqrt(dzdx * dzdx + dzdy * dzdy)) * 180 / 3.14159;
    }
}
"""

# Compile the kernel
mod = SourceModule(kernel_code)
calculate_slope = mod.get_function("calculate_slope")

In [ ]:
dsm_dataset = rio.open(dsm_file)

dsm_bounds = dsm_dataset.bounds
dsm_img = dsm_dataset.read(1)
print('The dsm bound is:', dsm_bounds)


# Prepare data
dsm = dsm_img.astype(np.float32)
height, width = dsm.shape
cell_size = 1.0  # Assuming cell size is 1 meter; adjust according to DSM metadata

# Allocate memory on the GPU
d_dsm = cuda.mem_alloc(dsm.nbytes)
d_slope = cuda.mem_alloc(dsm.nbytes)

# Copy DSM data to GPU
cuda.memcpy_htod(d_dsm, dsm)

In [ ]:
# Define block and grid size
block_size = (16, 16, 1)
grid_size = (int(np.ceil(width / block_size[0])), int(np.ceil(height / block_size[1])))

# Run the kernel
calculate_slope(d_dsm, d_slope, np.int32(width), np.int32(height), np.float32(cell_size), block=block_size, grid=grid_size)

In [ ]:
# Create an empty array for the result on the host
slope = np.empty_like(dsm)

# Copy result back to host
cuda.memcpy_dtoh(slope, d_slope)

# Free GPU memory
d_dsm.free()
d_slope.free()

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(slope, cmap="terrain")
plt.colorbar(label="Slope (degrees)")
plt.title("Slope Map")
plt.show()


## Step 4: The shadow kernel and a single test run

In [ ]:
# Define the CUDA kernel for the shadow calculation
kernel_shadow = """
#define PI 3.1415926

__global__ void calculate_shadow(float *dsm, bool *shadow, int width, int height, float cell_size, float sun_azimuth, float sun_elevation) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    // Check if the thread is within the bounds of the image
    if (x >= width || y >= height)
        return;

    int idx = y * width + x;

    // Convert sun azimuth and elevation to radians
    float azimuth_rad = sun_azimuth * 3.14159 / 180.0;
    float elevation_rad = sun_elevation * 3.14159 / 180.0;

    // Direction of the shadow ray based on sun azimuth
    float dx = sin(azimuth_rad);
    float dy = -cos(azimuth_rad); //because the azimuth start from the north, anti-clockwise
    float dz = tan(elevation_rad);

    // Initial height at the current pixel
    float initial_height = dsm[idx];
    bool in_shadow = false;

    // Trace the shadow ray
    for (float t = cell_size; t < 2000.0f; t += cell_size) { // Limit tracing distance
        int x_offset = x + int(dx * t / cell_size);
        int y_offset = y + int(dy * t / cell_size);

        if (x_offset < 0 || x_offset >= width || y_offset < 0 || y_offset >= height)
            break;  // Out of bounds

        int offset_idx = y_offset * width + x_offset;

        // Calculate height along the ray
        float height_along_ray = initial_height + dz * t;

        // Check if there is a higher point along the path
        if (dsm[offset_idx] > height_along_ray) {
            in_shadow = true;
            break;
        }
    }

    // Store shadow result: 1 for shadow, 0 for no shadow
    shadow[idx] = in_shadow;
}
"""

# Compile the kernel
mod = SourceModule(kernel_shadow)
calculate_shadows = mod.get_function("calculate_shadow")

In [ ]:
# Prepare parameters and allocate memory
height, width = dsm_img.shape

# Allocate memory for input DSM and output shadow map
d_dsm = cuda.mem_alloc(dsm_img.nbytes)
d_shadow = cuda.mem_alloc(dsm_img.nbytes)  # Shadow result (boolean)

# Copy DSM to GPU
cuda.memcpy_htod(d_dsm, dsm_img)

# Initialize shadow output array on CPU
shadow_result = np.zeros_like(dsm_img, dtype=bool)

In [ ]:
# Sun parameters (angles in degrees), you can get the sun positions using the tool of https://gml.noaa.gov/grad/solcalc/azel.html
# you can also find the Python module to do it.
sun_azimuth = 45.0       # Example: 45 degrees from the north
sun_elevation = 60.0     # Example: 30 degrees above the horizon
cell_size = 1.0          # Adjust based on DSM resolution, e.g., 1 meter per pixel


# Define block and grid size
block_size = (16, 16, 1)
grid_size = (int(np.ceil(width / block_size[0])), int(np.ceil(height / block_size[1])))


# Convert sun angles to float32 and launch the kernel
calculate_shadows(
    d_dsm, d_shadow, np.int32(width), np.int32(height),
    np.float32(cell_size), np.float32(sun_azimuth), np.float32(sun_elevation),
    block=block_size, grid=grid_size
)


In [ ]:
# Copy the shadow results back to the host
cuda.memcpy_dtoh(shadow_result, d_shadow)

# Free GPU memory
d_dsm.free()
d_shadow.free()

In [ ]:
binary_shadow = shadow_result.astype(np.uint8)  # 1 for True (shadow), 0 for False (non-shadow)


In [ ]:
shadow_result

In [ ]:
# Display the binary shadow image
plt.imshow(shadow_result)
# plt.colorbar(label="Shadow (1=True, 0=False)")
plt.title("Shadow Distribution")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(binary_shadow==0)
# plt.colorbar(label="Shadow (1=True, 0=False)")
# plt.title("Shadow Distribution")
plt.show()

## Step 5: Hourly shadow maps (8:00–17:00) combined into a GIF

### 1. Solar positions for the DSM centre

The DSM centre is projected from EPSG:2272 to WGS84, and the sun elevation/azimuth for each hour
8:00–17:00 comes from the NOAA solar calculator (https://gml.noaa.gov/grad/solcalc/azel.html).


In [ ]:
from pyproj import Transformer

# DSM center
center_x = (dsm_bounds.left + dsm_bounds.right) / 2
center_y = (dsm_bounds.bottom + dsm_bounds.top) / 2

# EPSG:2272 -> WGS84
transformer = Transformer.from_crs("EPSG:2272", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(center_x, center_y)

print("Latitude:", lat)
print("Longitude:", lon)


In [ ]:
import os

os.makedirs("shadow_maps", exist_ok=True)

solar_pos_lst = [
    (8, 12.89, 100.72),
    (9, 23.89, 111.46),
    (10, 34.04, 124.11),
    (11, 42.58, 139.89),
    (12, 48.38, 159.7),
    (13, 50.15, 182.62),
    (14, 47.35, 205.07),
    (15, 40.8, 223.93),
    (16, 31.79, 238.86),
    (17, 21.38, 250.96)
]


### 2. One shadow map per hour

In [ ]:
for hour, sun_elevation, sun_azimuth in solar_pos_lst:

    d_dsm = cuda.mem_alloc(dsm_img.nbytes)
    d_shadow = cuda.mem_alloc(dsm_img.nbytes)

    cuda.memcpy_htod(d_dsm, dsm_img)

    shadow_result = np.zeros_like(dsm_img, dtype=bool)

    calculate_shadows(
        d_dsm,
        d_shadow,
        np.int32(width),
        np.int32(height),
        np.float32(cell_size),
        np.float32(sun_azimuth),
        np.float32(sun_elevation),
        block=block_size,
        grid=grid_size
    )

    cuda.memcpy_dtoh(shadow_result, d_shadow)

    plt.figure(figsize=(8, 8))
    plt.imshow(shadow_result, cmap="gray")
    plt.title(f"Shadow Distribution - {hour}:00")
    plt.axis("off")
    plt.savefig(
        f"shadow_maps/shadow_{hour:02d}.png",
        bbox_inches="tight"
    )
    plt.close()

    d_dsm.free()
    d_shadow.free()


### 3. Combine the PNGs into a GIF

In [ ]:
from PIL import Image
import glob

files = sorted(glob.glob("shadow_maps/shadow_*.png"))

images = [Image.open(f) for f in files]

images[0].save(
    "shadow_analysis.gif",
    save_all=True,
    append_images=images[1:],
    duration=500,
    loop=0
)


In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename="shadow_analysis.gif"))


## 6 Discussion

The GIF shows the shadow footprint shrinking toward early afternoon and stretching again toward the ends
of the day. At 8:00 the sun is low in the east-south-east (elevation 12.9°, azimuth 100.7°), so shadows
are longest and point west-north-west; at 13:00 the sun peaks at 50.2° just south of due west–east
(azimuth 182.6°) and the shadow footprint is smallest, close to the buildings' own outlines; by 17:00 the
sun sits in the west-south-west (21.4°, azimuth 251.0°) and shadows stretch toward east-north-east. The
morning/evening pattern is not symmetric because the solar maximum falls just after 13:00 on the study
day.

Two implementation details matter. First, azimuth is measured from north and anti-clockwise, so the row
direction of the ray is `dy = -cos(azimuth)` — the course notebook fixed this sign in its Oct 2025 update
(the original `+cos` flipped the shadow direction). Second, every pixel walks up to 2000 one-metre steps
along its ray, i.e. roughly 5.3 M pixels × up to 2000 samples per hour; this is what makes the GPU
worthwhile — the ten maps render in seconds on a T4, while a pure-Python version would take hours.

The only change to the course notebook is the data entry point: the DSM is fetched with `gdown`
(md5 `4f48c024da95b4337dda37804f02c570`, identical to the Drive file) instead of a Drive mount, so the
notebook is self-contained and every run regenerates the same GIF.
